# Store in ChromaDB — standalone

This notebook is **fully self-contained**. It only needs `chunks_cache.json` (written by `ingest.ipynb`).  
Run it in a **fresh kernel** — no imports or variables carried over from the ingest notebook.

In [6]:
import json
import chromadb
import logging

logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)

CACHE_FILE      = "./chunks_cache.json"
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "aethon_kb"

# ── 1. Load cache ─────────────────────────────────────────────────────────────
with open(CACHE_FILE, encoding="utf-8") as f:
    cache = json.load(f)

all_chunks  = [{k: v for k, v in c.items() if k != "embedding"} for c in cache]
embeddings  = [c["embedding"] for c in cache]

print(f"Loaded {len(all_chunks)} chunks  |  dim={len(embeddings[0])}")

# ── 2. Connect to Chroma ──────────────────────────────────────────────────────
chroma = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"Chroma connected  |  existing collections: {chroma.list_collections()}")

try:
    chroma.delete_collection(COLLECTION_NAME)
    print(f"Deleted old '{COLLECTION_NAME}'")
except Exception:
    pass

collection = chroma.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
    embedding_function=None,
)

# ── 3. Insert in small batches (avoids any single large allocation) ───────────
BATCH = 10
for i in range(0, len(all_chunks), BATCH):
    sl = slice(i, i + BATCH)
    collection.add(
        ids        = [c["chunk_id"]    for c in all_chunks[sl]],
        documents  = [c["text"]        for c in all_chunks[sl]],
        embeddings = embeddings[sl],
        metadatas  = [
            {"filename": c["filename"], "chunk_index": c["chunk_index"], "topic": c["topic"]}
            for c in all_chunks[sl]
        ],
    )
    print(f"  Inserted batch {i//BATCH + 1}  ({min(i+BATCH, len(all_chunks))}/{len(all_chunks)} chunks)")

print(f"\nDone — {collection.count()} chunks stored in '{COLLECTION_NAME}'")
print(f"ChromaDB at: {CHROMA_PATH}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Loaded 31 chunks  |  dim=1536
Chroma connected  |  existing collections: [Collection(name=aethon_kb)]
Deleted old 'aethon_kb'
  Inserted batch 1  (10/31 chunks)
  Inserted batch 2  (20/31 chunks)
  Inserted batch 3  (30/31 chunks)
  Inserted batch 4  (31/31 chunks)

Done — 31 chunks stored in 'aethon_kb'
ChromaDB at: ./chroma_db


## Smoke test
Embed a query with OpenAI and retrieve from Chroma. Confirms the whole pipeline end-to-end.

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path=os.path.join("..", ".env"))
oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def embed(text):
    return oai.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

def query(question, n=3):
    results = collection.query(
        query_embeddings=[embed(question)],
        n_results=n,
        include=["documents", "metadatas", "distances"],
    )
    print(f"Query: '{question}'\n")
    for rank, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), 1
    ):
        print(f"  [{rank}] {meta['topic']}  ({meta['filename']})  dist={dist:.4f}")
        print(f"       {doc[:180]}...")
        print()

query("Who is the CTO of Aethon Dynamics?")
query("What is VoltCore and how is it priced?")
query("What was Aethon's ARR in 2024?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'Who is the CTO of Aethon Dynamics?'

  [1] Document Overview  (employees.md)  dist=0.3163
       # Aethon Dynamics — Employees

This document covers the leadership team and selected key employees at Aethon Dynamics....

  [2] Dr. Maya Krishnan  (employees.md)  dist=0.3234
       ## Leadership Team

### Dr. Maya Krishnan — Chief Executive Officer
Maya co-founded Aethon Dynamics in 2016. Before Aethon, she spent eight years as a power systems engineer at a r...

  [3] Company Overview  (company_overview.md)  dist=0.3476
       # Aethon Dynamics — Company Overview

## About

Aethon Dynamics is a clean-energy technology company that builds software and hardware for electrical grid optimization and battery ...

Query: 'What is VoltCore and how is it priced?'

  [1] VoltCore Product Details  (products.md)  dist=0.2285
       ## VoltCore — Battery Storage Optimization

VoltCore is the optimization engine that decides when to charge and discharge battery storage systems to maximize val